# Silver Layer — CRM Customer Info
Clean and normalize `crm_cust_info`.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "silver",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
VOLUME = os.environ.get("CLICKZETTA_VOLUME", "medallion_vol")

## Read Bronze Table

In [ ]:
df = session.table(f"bronze.crm_cust_info")

## Silver Transformations

### Trimming

In [ ]:
from clickzetta.zettapark.types import StringType
from clickzetta.zettapark import functions as F

for field in df.schema.fields:
    if isinstance(field.datatype, StringType):
        df = df.with_column(field.name, F.trim(F.col(field.name)))

### Normalization

In [ ]:
df = (
    df
    .with_column(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .with_column(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

### Remove Records with Missing Customer ID

In [ ]:
df = df.filter(F.col("cst_id").is_not_null())

### Rename Columns

In [ ]:
RENAME_MAP = {
    "cst_id":             "customer_id",
    "cst_key":            "customer_number",
    "cst_firstname":      "first_name",
    "cst_lastname":       "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr":           "gender",
    "cst_create_date":    "created_date",
}
for old, new in RENAME_MAP.items():
    df = df.with_column_renamed(old, new)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Silver Table

In [ ]:
df.write.save_as_table(f"silver.crm_customers", mode="overwrite")
print("crm_customers OK")

## Verify

In [ ]:
session.table(f"silver.crm_customers").limit(5).show()